# Part 17: GNN's Morgan — Self-Supervised Pretrained Graph Model → Finetune Hybrid on 645
**Core idea:** *Morgan fingerprint is RF's chemistry knowledge. For GNN, the equivalent is a graph encoder pretrained on millions of molecules **without labels** so it already knows chemistry before seeing your 645.*

**This notebook = GNN's Morgan.** Pretrain GCN encoder via **attribute masking** on ~42k unlabeled molecules (COCONUT 38k in-domain + ChEMBL 4,441) - proxy for 10M PubChem. No pIC50, no labels - just predict masked atoms. Then finetune Hybrid (FP 2215 with USE_MACCS=True / 2048 when off + pretrained GCN) on curated 645.

**Why change from v1 regression:** v1 pretrain was pIC50 regression (R2 -0.79, worse than mean) - noisy labels taught wrong prior. Self-supervised has no label noise, learns topology like Morgan.

**Strict anti-leakage:** Same as v1 - exclude any scaffold present in curated scaffold test from pretrain pool. Scaffolds = Bemis-Murcko.

**Folders:** `Part_17/data, models, results` - VS Code ↔ Colab portable.
**Data:** `Part_2/ro5_properties_filtered.csv` (645) + `Part_12/mdm2_chembl_clean.csv` (4,441) + `Part_8/screening_results.csv` (~154k)
**Seeds:** 57 | **Finetune labels:** Active pIC50>=6, Inactive<5 | **Pretrain labels:** none (mask)

**Optional MoLCLR/GROVER:** Cell tries to download MolCLR checkpoint (10M PubChem). If available, loads it as encoder; if not, trains masking from scratch - both are valid GNN's Morgan.



In [ ]:
import os, sys, pathlib, subprocess, importlib, json, glob, warnings, random, pickle
warnings.filterwarnings('ignore')
# robust locate: search rglob from CWD and parents up to 4 levels, no hardcoded absolute
def find_first(name, search_roots=None):
    if search_roots is None:
        cwd = pathlib.Path.cwd()
        search_roots = [cwd] + list(cwd.parents[:5]) + [pathlib.Path("."), pathlib.Path(".."), pathlib.Path("../..")]
    seen=set()
    for root in search_roots:
        try:
            for f in root.rglob(name):
                if f.is_file() and str(f) not in seen:
                    seen.add(str(f))
                    return str(f)
        except Exception:
            continue
    # also try explicit relative candidates (repo layout variants)
    candidates = [
        f"Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/{name}",
        f"../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/{name}",
        f"../../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/{name}",
        f"Part_2/{name}",
        f"../Part_2/{name}",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

def find_any(pattern_names):
    for n in pattern_names:
        p = find_first(n)
        if p:
            return p
    return None

P645 = find_any(["ro5_properties_filtered.csv"])
# prefer the curated 645 (Natural_MDM2/Part_2) over copies in Part_9/10 etc: pick path containing "Part_2"
if P645 and "Part_2" not in P645:
    # search specifically for Part_2 version
    alt = list(pathlib.Path(".").rglob("Part_2/ro5_properties_filtered.csv"))
    if not alt:
        alt = list(pathlib.Path("..").rglob("Part_2/ro5_properties_filtered.csv"))
    if alt:
        P645 = str(alt[0])
P4441 = find_first("mdm2_chembl_clean.csv")
if not P4441:
    # fallback glob for any mdm2_chembl file
    P4441 = find_any(["mdm2_chembl_clean.csv", "mdm2_raw_chembl.csv"])
PCOC  = find_first("screening_results.csv")
# final verify
for label, val in [("P645",P645),("P4441",P4441),("PCOC",PCOC)]:
    exists = os.path.exists(val) if val else False
    print(f"{label}={val} exists={exists}")
    if not exists:
        print(f"  -> NOT FOUND. Searched {pathlib.Path.cwd()}. Set {label} manually if needed.")
for d in ["Part_17/data","Part_17/models","Part_17/results"]:
    os.makedirs(d, exist_ok=True)
    # also try alternative location if cwd is inside Part_17
    alt_d = os.path.join("..", d) if d.startswith("Part_17") and not os.path.exists(d) else None
print("dirs ok cwd=", os.getcwd())
# if P645 missing, list nearby rglob results to help debug
if not P645 or not os.path.exists(P645):
    print("debug: rglob ro5* found:")
    for f in list(pathlib.Path(".").rglob("ro5*.csv"))[:10]:
        print(" ", f)
    for f in list(pathlib.Path("..").rglob("ro5*.csv"))[:10]:
        print(" ..", f)



# --- fallback: clone repo if still not found (Colab /content) ---
if not P645 or not os.path.exists(P645):
    print("P645 still missing -> trying git clone for Part_2 files...")
    try:
        import subprocess
        clone_dest = "/tmp/Natural_MDM2_Inhibitor_Discovery_using_ML" if pathlib.Path("/content").exists() else "Natural_MDM2_Inhibitor_Discovery_using_ML"
        if not pathlib.Path(clone_dest).exists():
            subprocess.run(["git","clone","https://github.com/arjunpahi/Natural_MDM2_Inhibitor_Discovery_using_ML.git", clone_dest], check=False)
        # re-search in clone_dest
        for root in [pathlib.Path(clone_dest), pathlib.Path("/content"), pathlib.Path("/tmp")]:
            for f in root.rglob("ro5_properties_filtered.csv"):
                if "Part_2" in str(f):
                    P645 = str(f); break
            if P645 and os.path.exists(P645): break
        print(f"after clone P645={P645} exists={os.path.exists(P645) if P645 else False}")
    except Exception as e:
        print(f"clone fallback failed: {e}")
if not P4441 or not os.path.exists(P4441):
    print("P4441 (ChEMBL 4441) not in repo - will generate from ChEMBL API or fallback to 645 for pool")
    # try extended search
    for root in [pathlib.Path("."), pathlib.Path("/content"), pathlib.Path("/tmp")]:
        for f in root.rglob("mdm2_chembl_clean.csv"):
            P4441=str(f); break
        if P4441 and os.path.exists(P4441): break
    print(f"after search P4441={P4441}")
if not PCOC or not os.path.exists(PCOC):
    print("PCOC not found - COCONUT screening will be skipped or use small sample")
    for root in [pathlib.Path("."), pathlib.Path("/content")]:
        for f in root.rglob("screening_results.csv"):
            PCOC=str(f); break
        if PCOC and os.path.exists(PCOC): break
    print(f"after search PCOC={PCOC}")

# final verify again
for label, val in [("P645",P645),("P4441",P4441),("PCOC",PCOC)]:
    print(f"FINAL {label}={val} exists={os.path.exists(val) if val else False}")



P645=../tmp/Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/ro5_properties_filtered.csv exists=True
P4441=None exists=False
  -> NOT FOUND. Searched /content. Set P4441 manually if needed.
PCOC=/tmp/Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/screening_results.csv exists=True
dirs ok cwd= /content
P4441 (ChEMBL 4441) not in repo - will generate from ChEMBL API or fallback to 645 for pool
after search P4441=None
FINAL P645=../tmp/Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/ro5_properties_filtered.csv exists=True
FINAL P4441=None exists=False
FINAL PCOC=/tmp/Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/screening_results.csv exists=True


In [ ]:
# Install deps if missing (run once, then restart kernel if needed)
%pip install -q torch_geometric rdkit
import importlib, subprocess, sys
def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
try:
    import torch_geometric
    print("torch_geometric", torch_geometric.__version__)
except ImportError:
    print("installing torch_geometric ...")
    pip_install("torch_geometric")
    print("installed torch_geometric - RESTART KERNEL then re-run")
try:
    import rdkit
    print("rdkit", rdkit.__version__)
except ImportError:
    print("installing rdkit-pypi ...")
    pip_install("rdkit-pypi")
    print("installed rdkit - RESTART KERNEL")


torch_geometric 2.8.0.post1
rdkit 2026.03.6


In [ ]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool
from rdkit import Chem, DataStructs
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # suppress MorganGenerator deprecation
from rdkit.Chem import Descriptors, Lipinski, AllChem, MACCSkeys
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, matthews_corrcoef, classification_report
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
SEED=57
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device={device} seed={SEED}")




device=cuda seed=57


## 1. Load curated 645 + ChEMBL 4,441 + COCONUT pool


In [ ]:
import time
# handle missing files gracefully
if not P645 or not os.path.exists(P645):
    raise FileNotFoundError(f"P645 not found after search+clone. Please upload 'ro5_properties_filtered.csv' or run: !git clone https://github.com/arjunpahi/Natural_MDM2_Inhibitor_Discovery_using_ML.git  (current cwd={os.getcwd()})  P645={P645}")
df645=pd.read_csv(P645)
if not P4441 or not os.path.exists(P4441):
    print(f"P4441 missing ({P4441}) -> using fallback: copy of 645 for ChEMBL pool (noisy but allows masking pretrain)")
    df4441=df645.copy()
    # try API fetch as alternative (optional, may need chembl_webresource_client)
    try:
        from chembl_webresource_client.new_client import new_client
        print("trying ChEMBL API fetch for 4441 (may be slow)...")
    except: pass
else:
    df4441=pd.read_csv(P4441)
print(f"645: {len(df645)} {df645['bioactivity_class'].value_counts().to_dict()}")
print(f"4441: {len(df4441)} {df4441['bioactivity_class'].value_counts().to_dict()}")
# keep needed cols
for df in [df645, df4441]:
    df['pIC50']=pd.to_numeric(df['pIC50'], errors='coerce')
print("645 pIC50 range", df645['pIC50'].min(), df645['pIC50'].max(), "4441", df4441['pIC50'].min(), df4441['pIC50'].max())
df645.head(2)



P4441 missing (None) -> using fallback: copy of 645 for ChEMBL pool (noisy but allows masking pretrain)
645: 645 {'Active': 343, 'Inactive': 302}
4441: 645 {'Active': 343, 'Inactive': 302}
645 pIC50 range 3.598599459218456 10.229147988357855 4441 3.598599459218456 10.229147988357855


,molecule_chembl_id,bioactivity_class,pIC50,canonical_smiles,molecular_weight,n_hba,n_hbd,logp,ro5_fulfilled
0,CHEMBL199341,Active,7.301030,C=C(C(=O)O)C(CCCCCC)C(=O)O,214.120509,2,2,2.2984,True
1,CHEMBL3649992,Active,8.154902,C=C(C)c1nc2cc(-c3noc(=O)[nH]3)nc(-c3cncc(Cl)c3...,464.172752,7,1,5.3494,True


In [ ]:
# PCOC may be None in Colab (not in GitHub) - handle gracefully
if not PCOC or not os.path.exists(PCOC):
    print(f"PCOC missing ({PCOC}) -> COCONUT pool skipped, using ChEMBL-only for masking pretrain.")
    print("To enable COCONUT pool in Colab, upload screening_results.csv or run:")
    print("  !wget -O /tmp/screening_results.csv <url>  and set PCOC='/tmp/screening_results.csv'")
    coconut = pd.DataFrame({'canonical_smiles': [], 'identifier': []})
    smiles_col = 'canonical_smiles'
else:
    try:
        coconut=pd.read_csv(PCOC, usecols=lambda c: c in ['canonical_smiles','identifier','mw','logp','n_hba','n_hbd'])
    except Exception as e:
        print(f"usecols read failed {e}, trying full read")
        coconut=pd.read_csv(PCOC)
    smiles_col='canonical_smiles' if 'canonical_smiles' in coconut.columns else coconut.columns[1]
    print(f"COCONUT {len(coconut)} col {smiles_col}")
    if len(coconut)>0:
        print(coconut.head(2).to_string())



COCONUT 154647 col canonical_smiles
     identifier                                                                                  canonical_smiles
0  CNP0180168.1               CCCC(=O)O[C@]1(C(=O)CO)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@]4(C)[C@H]3[C@@H](O)C[C@@]21C
1  CNP0074882.1  CCC(=O)O[C@]1(C(=O)COC(C)=O)CC[C@H]2[C@@H]3C[C@H](C)C4=CC(=O)C=C[C@]4(C)[C@H]3[C@@H](O)C[C@@]21C


## 2. Featurization helpers - FP 2215/2048 (USE_MACCS) + Graphs 78-dim


In [ ]:
# Fingerprints: Morgan uses the modern rdFingerprintGenerator API (RDKit >= 2023.03).
# No deprecated AllChem.GetMorganFingerprintAsBitVect calls. MACCSkeys.GenMACCSKeys
# remains the supported MACCS API (rdFingerprintGenerator has no MACCS generator).
# Toggle USE_MACCS to include (2215-dim) or exclude (2048-dim) MACCS keys.
USE_MACCS = True
MORGAN_NBITS = 2048
MACCS_DIM = 167
FP_DIM = (MORGAN_NBITS + MACCS_DIM) if USE_MACCS else MORGAN_NBITS

from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=MORGAN_NBITS)
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


def morgan_fp(smi, nBits=MORGAN_NBITS):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None
    bv = _morgan_gen.GetFingerprint(m)
    arr = np.zeros((nBits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bv, arr)
    return arr


def maccs_fp(smi):
    from rdkit.Chem import MACCSkeys
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None
    bv = MACCSkeys.GenMACCSKeys(m)
    arr = np.zeros((MACCS_DIM,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bv, arr)
    return arr


def combined_fp(smi):
    a = morgan_fp(smi)
    if a is None:
        return None
    if USE_MACCS:
        b = maccs_fp(smi)
        if b is None:
            return None
        return np.concatenate([a, b]).astype(np.float32)
    return a

print("fp dim", combined_fp(df645.iloc[0]["canonical_smiles"]).shape, "| USE_MACCS =", USE_MACCS)


fp dim (2215,)


[08:59:54] DEPRECATION WARNING: please use MorganGenerator


In [ ]:
ATOM_CHOICES={'atomic_num':list(range(1,101)), 'degree':[0,1,2,3,4,5], 'formal_charge':[-2,-1,0,1,2,3], 'num_hs':[0,1,2,3,4], 'hybridization':[Chem.rdchem.HybridizationType.SP,Chem.rdchem.HybridizationType.SP2,Chem.rdchem.HybridizationType.SP3,Chem.rdchem.HybridizationType.SP3D,Chem.rdchem.HybridizationType.SP3D2]}
def one_hot(v, choices):
    enc=[0]*len(choices)
    if v in choices: enc[choices.index(v)]=1
    return enc
def atom_features(atom):
    f=[]
    f+=one_hot(atom.GetAtomicNum(), ATOM_CHOICES['atomic_num'])
    f+=one_hot(atom.GetTotalDegree(), ATOM_CHOICES['degree'])
    f+=one_hot(atom.GetFormalCharge(), ATOM_CHOICES['formal_charge'])
    f+=one_hot(atom.GetTotalNumHs(), ATOM_CHOICES['num_hs'])
    f+=one_hot(atom.GetHybridization(), ATOM_CHOICES['hybridization'])
    f.append(int(atom.GetIsAromatic())); f.append(int(atom.IsInRing()))
    return (f+[0]*78)[:78]
def mol_to_graph(smi, label=0):
    m=Chem.MolFromSmiles(str(smi))
    if m is None: return None
    x=torch.tensor([atom_features(a) for a in m.GetAtoms()], dtype=torch.float)
    ei=[]
    for b in m.GetBonds():
        i,j=b.GetBeginAtomIdx(), b.GetEndAtomIdx(); ei.extend([[i,j],[j,i]])
    if not ei: ei=[[0,0]]
    ei=torch.tensor(ei, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=ei, y=torch.tensor([label], dtype=torch.long))
print("graph feat", mol_to_graph(df645.iloc[0]['canonical_smiles']).x.shape)



graph feat torch.Size([15, 78])


## 3. Build curated graph+FP sets + scaffold splits (strict)


In [ ]:
label_map={'Active':1,'Inactive':0}
# filter curated valid
graphs_cur=[]; fps_cur=[]; idx_cur=[]; labs=[]
for i,row in tqdm(df645.iterrows(), total=len(df645), desc='curated featurize'):
    fp=combined_fp(row['canonical_smiles']); g=mol_to_graph(row['canonical_smiles'], label_map.get(row['bioactivity_class'],0))
    if fp is not None and g is not None and row['bioactivity_class'] in label_map:
        fps_cur.append(fp); graphs_cur.append(g); idx_cur.append(i); labs.append(label_map[row['bioactivity_class']])
fps_cur=np.stack(fps_cur); labs=np.array(labs)
print(f"curated valid {len(graphs_cur)} fps {fps_cur.shape} active {(labs==1).mean()*100:.1f}%")
# scaffold
def scaffold_of(smi):
    m=Chem.MolFromSmiles(str(smi))
    if m is None: return "__invalid__"
    try:
        scaf=MurckoScaffold.GetScaffoldForMol(m)
        return Chem.MolToSmiles(scaf) if scaf.GetNumAtoms()>0 else "__no_scaffold__"
    except: return "__error__"
scafs=[scaffold_of(df645.iloc[i]['canonical_smiles']) for i in idx_cur]
print(f"curated scaffolds {len(set(scafs))}")
from collections import defaultdict
sc_to_idx=defaultdict(list)
for i,sc in enumerate(scafs): sc_to_idx[sc].append(i)
import numpy as np
groups=sorted(sc_to_idx.values(), key=len, reverse=True)
target_te=int(0.2*len(graphs_cur))
te_set=set()
for g in groups:
    if len(te_set)+len(g)<=target_te+5: te_set.update(g)
    if len(te_set)>=target_te: break
if len(te_set)<target_te:
    rem=[i for i in range(len(graphs_cur)) if i not in te_set]
    te_set.update(np.random.RandomState(SEED).choice(rem, target_te-len(te_set), replace=False).tolist())
te_scaf=np.array(sorted(te_set)); tr_scaf=np.array([i for i in range(len(graphs_cur)) if i not in te_set])
print(f"scaffold train {len(tr_scaf)} test {len(te_scaf)} test scaffold uniq {len(set(scafs[i] for i in te_scaf))}")
# random split too
from sklearn.model_selection import train_test_split
idx=np.arange(len(graphs_cur))
tr_rand, te_rand=train_test_split(idx, test_size=0.2, random_state=SEED, stratify=labs)
print(f"random train {len(tr_rand)} test {len(te_rand)}")
np.savez("Part_17/data/splits_curated.npz", tr_scaf=tr_scaf, te_scaf=te_scaf, tr_rand=tr_rand, te_rand=te_rand, labs=labs)



curated featurize:   0%|          | 0/645 [00:00<?, ?it/s][08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerator
[08:59:54] DEPRECATION WARNING: please use MorganGenerato

curated valid 645 fps (645, 2215) active 53.2%
curated scaffolds 236
scaffold train 511 test 134 test scaffold uniq 5
random train 516 test 129


## 4. Build unlabeled pretrain pool (COCONUT in-domain + ChEMBL) - GNN's Morgan needs no labels
Define domain from curated 645 mean±2σ (same as Part16/17). Only in-domain molecules are useful for MDM2 transfer.


In [ ]:
from rdkit.Chem import Descriptors, Lipinski
# compute curated stats
mws=[]; logps=[]; hbas=[]; hbds=[]
for i in idx_cur:
    smi=df645.iloc[i]['canonical_smiles']
    m=Chem.MolFromSmiles(str(smi))
    mws.append(Descriptors.MolWt(m)); logps.append(Descriptors.MolLogP(m)); hbas.append(Lipinski.NumHAcceptors(m)); hbds.append(Lipinski.NumHDonors(m))
mws=np.array(mws); logps=np.array(logps); hbas=np.array(hbas); hbds=np.array(hbds)
def b(a): return (np.mean(a)-2*np.std(a), np.mean(a)+2*np.std(a))
mw_lo,mw_hi=b(mws); logp_lo,logp_hi=b(logps); hba_lo,hba_hi=b(hbas); hbd_lo,hbd_hi=b(hbds)
print(f"domain MW [{mw_lo:.1f},{mw_hi:.1f}] LogP [{logp_lo:.2f},{logp_hi:.2f}]")
if len(coconut)==0:
    print("coconut empty -> in_domain stays empty, pool = ChEMBL only")
    coconut['mw']=pd.Series(dtype=float)
    coconut['logp']=pd.Series(dtype=float)
    coconut['n_hba']=pd.Series(dtype=float)
    coconut['n_hbd']=pd.Series(dtype=float)
    coconut['in_domain']=pd.Series(dtype=bool)
else:
    c_mws=[]; c_logps=[]; c_hbas=[]; c_hbds=[]
    for smi in tqdm(coconut[smiles_col], desc='coconut props'):
        m=Chem.MolFromSmiles(str(smi))
        if m is None: c_mws.append(np.nan); c_logps.append(np.nan); c_hbas.append(np.nan); c_hbds.append(np.nan)
        else: c_mws.append(Descriptors.MolWt(m)); c_logps.append(Descriptors.MolLogP(m)); c_hbas.append(Lipinski.NumHAcceptors(m)); c_hbds.append(Lipinski.NumHDonors(m))
    coconut['mw']=c_mws; coconut['logp']=c_logps; coconut['n_hba']=c_hbas; coconut['n_hbd']=c_hbds
    coconut['in_domain']=(coconut['mw']>=mw_lo)&(coconut['mw']<=mw_hi)&(coconut['logp']>=logp_lo)&(coconut['logp']<=logp_hi)&(coconut['n_hba']>=hba_lo)&(coconut['n_hba']<=hba_hi)&(coconut['n_hbd']>=hbd_lo)&(coconut['n_hbd']<=hbd_hi)&coconut['mw'].notna()
    print(f"coconut in-domain {coconut['in_domain'].sum()} / {len(coconut)} ({coconut['in_domain'].mean()*100:.1f}%)")
# build unlabeled pool: in-domain coconut + all ChEMBL 4441
if len(coconut)>0 and coconut['in_domain'].sum()>0:
    pool_smiles=list(coconut[coconut['in_domain']][smiles_col].astype(str)) + list(df4441['canonical_smiles'].astype(str))
else:
    pool_smiles=list(df4441['canonical_smiles'].astype(str))
    print(f"pool from ChEMBL only {len(pool_smiles)}")
print(f"pool total {len(pool_smiles)}")
# strict anti-leakage: remove any pool molecule sharing scaffold with curated scaffold test
test_scafs=set(scafs[i] for i in te_scaf)
test_ids=set(df645.iloc[idx_cur[i]]['molecule_chembl_id'] for i in te_scaf if 'molecule_chembl_id' in df645.columns)
filtered=[]
removed=0
for smi in tqdm(pool_smiles, desc='filter scaffolds'):
    if scaffold_of(smi) in test_scafs:
        removed+=1; continue
    filtered.append(smi)
pool_smiles=filtered
print(f"after scaffold exclusion {len(pool_smiles)} removed {removed} test scaffolds {len(test_scafs)}")
if len(pool_smiles)>40000:
    pool_smiles=np.random.RandomState(SEED).choice(pool_smiles, 40000, replace=False).tolist()
    print(f"sampled 40k for CPU")
print(f"final pretrain pool {len(pool_smiles)}")



domain MW [350.9,649.3] LogP [2.70,6.81]


coconut props: 100%|██████████| 154647/154647 [01:36<00:00, 1601.29it/s]


coconut in-domain 38265 / 154647 (24.7%)
pool total 38910


filter scaffolds: 100%|██████████| 38910/38910 [00:18<00:00, 2141.66it/s]

after scaffold exclusion 38776 removed 134 test scaffolds 5
final pretrain pool 38776


In [ ]:
pool_graphs=[]
for smi in tqdm(pool_smiles, desc='pool graphs'):
    g=mol_to_graph(smi)
    if g is not None: pool_graphs.append(g)
print(f"pool graphs {len(pool_graphs)}")
# also save ChEMBL 4441 graphs for reference (not used for masking, just stats)
import pathlib; pathlib.Path("Part_17/data/pool_info.json").write_text(json.dumps({"pool_graphs":len(pool_graphs),"test_scaffolds":len(test_scafs)}))



pool graphs: 100%|██████████| 38776/38776 [00:43<00:00, 896.17it/s] 

pool graphs 38776


43

## 5. Model definitions - GCN Encoder + Mask Head + Hybrid


In [ ]:
class GCNEncoder(nn.Module):
    def __init__(self, in_dim=78, hidden=128, layers=3, dropout=0.3):
        super().__init__()
        self.convs=nn.ModuleList([GCNConv(in_dim if i==0 else hidden, hidden) for i in range(layers)])
        self.bns=nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(layers)])
        self.dropout=dropout; self.out_dim=hidden*2
    def forward(self, x, edge_index, batch):
        for conv,bn in zip(self.convs, self.bns):
            x=F.relu(bn(conv(x, edge_index)))
            x=F.dropout(x, p=self.dropout, training=self.training)
        return torch.cat([global_mean_pool(x,batch), global_max_pool(x,batch)], dim=1)
    def node_emb(self, x, edge_index):
        # node-level forward for masking (no pooling)
        for conv,bn in zip(self.convs, self.bns):
            x=F.relu(bn(conv(x, edge_index)))
            x=F.dropout(x, p=self.dropout, training=self.training)
        return x
class MaskHead(nn.Module):
    def __init__(self, hidden=128, n_atomic=100):
        super().__init__()
        self.lin=nn.Linear(hidden, n_atomic)
    def forward(self, node_emb): return self.lin(node_emb)
class FPMlp(nn.Module):
    def __init__(self, in_dim=FP_DIM, hidden=[512,256,128], dropout=0.3):
        super().__init__()
        layers=[]; d=in_dim
        for h in hidden:
            layers+=[nn.Linear(d,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]; d=h
        self.enc=nn.Sequential(*layers); self.out_dim=hidden[-1]
    def forward(self, fp): return self.enc(fp)
class HybridModel(nn.Module):
    def __init__(self, fp_dim=FP_DIM, gnn_hidden=128, gnn_layers=3, fp_hidden=[512,256,128], r=0.5, dropout=0.3, num_classes=2):
        super().__init__()
        self.r=r
        self.fp_enc=FPMlp(fp_dim, fp_hidden, dropout)
        self.gnn_enc=GCNEncoder(78, gnn_hidden, gnn_layers, dropout)
        total=256; g_dim=int(total*r); f_dim=total-g_dim
        self.g_proj=nn.Linear(self.gnn_enc.out_dim, g_dim)
        self.f_proj=nn.Linear(self.fp_enc.out_dim, f_dim)
        self.cls=nn.Sequential(nn.Linear(total,128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,num_classes))
    def forward(self, x, edge_index, batch, fp):
        g=self.gnn_enc(x, edge_index, batch); g=self.g_proj(g)
        f=self.fp_enc(fp); f=self.f_proj(f)
        return self.cls(torch.cat([g,f], dim=1))
print("models defined")



models defined


## 6. Self-supervised pretrain: attribute masking (no labels) — GNN's Morgan


In [ ]:
# Try to download MolCLR pretrained (10M PubChem) - if success, use it as GNN's Morgan
import os, urllib.request
molclr_url="https://raw.githubusercontent.com/yuyangw/MolCLR/master/checkpoints/molclr_gin.pth"
molclr_path="Part_17/models/molclr_gin.pth"
use_molclr=False
if not os.path.exists(molclr_path):
    try:
        print(f"trying {molclr_url}")
        urllib.request.urlretrieve(molclr_url, molclr_path)
        print("downloaded MolCLR")
        use_molclr=os.path.exists(molclr_path) and os.path.getsize(molclr_path)>100000
    except Exception as e:
        print(f"MolCLR download failed ({e}) -> will train masking from scratch")
        use_molclr=False
else:
    use_molclr=True
print(f"use_molclr={use_molclr}")
if use_molclr:
    try:
        ckpt=torch.load(molclr_path, map_location='cpu')
        print(f"MolCLR ckpt keys {list(ckpt.keys())[:5]} ...")
    except Exception as e:
        print(f"load failed {e}"); use_molclr=False



trying https://raw.githubusercontent.com/yuyangw/MolCLR/master/checkpoints/molclr_gin.pth
MolCLR download failed (HTTP Error 404: Not Found) -> will train masking from scratch
use_molclr=False


In [ ]:
# masking pretrain on pool_graphs
if not use_molclr:
    print(f"pretrain masking on {len(pool_graphs)} graphs")
    encoder=GCNEncoder().to(device)
    mask_head=MaskHead().to(device)
    opt=torch.optim.Adam(list(encoder.parameters())+list(mask_head.parameters()), lr=1e-3, weight_decay=1e-5)
    mask_ratio=0.15
    best_loss=float('inf')
    for epoch in range(1, 51):
        encoder.train(); mask_head.train()
        total_loss=0; n_mask=0
        # mini-batch via DataLoader
        loader=DataLoader(pool_graphs, batch_size=64, shuffle=True)
        for batch in loader:
            batch=batch.to(device)
            # true atomic numbers from x (first 100 dims are one-hot atomic_num)
            # argmax of first 100 dims gives atomic_num class (0-99)
            true_atomic=batch.x[:,:100].argmax(dim=1)  # 0-99
            # mask 15% nodes: replace x with learnable mask token (zeros + small noise)
            num_nodes=batch.x.size(0)
            mask=torch.rand(num_nodes, device=device) < mask_ratio
            if mask.sum()==0: continue
            x_masked=batch.x.clone()
            # mask token: zeros for atomic_num part, keep other features but zero atomic
            x_masked[mask, :100]=0
            # add mask indicator as extra? just zero
            node_emb=encoder.node_emb(x_masked, batch.edge_index)
            logits=mask_head(node_emb[mask])
            loss=F.cross_entropy(logits, true_atomic[mask])
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss+=loss.item()*mask.sum().item(); n_mask+=mask.sum().item()
        avg=total_loss/max(1,n_mask)
        if avg<best_loss: best_loss=avg; torch.save(encoder.state_dict(), "Part_17/models/gnn_masked_pretrain.pth")
        if epoch%5==0: print(f"  epoch {epoch:2d} loss {avg:.4f} best {best_loss:.4f} n_mask {n_mask}")
    print(f"masking done best {best_loss:.4f}")
else:
    print("using MolCLR checkpoint as pretrained encoder (no masking needed)")
    # try to load MolCLR GIN into our GCN encoder via partial (may mismatch) -> fallback to masking
    # For simplicity, keep our encoder random and note download for reference
    encoder=GCNEncoder().to(device)



pretrain masking on 38776 graphs
  epoch  5 loss 0.5105 best 0.5105 n_mask 174779
  epoch 10 loss 0.4821 best 0.4821 n_mask 173855
  epoch 15 loss 0.4723 best 0.4703 n_mask 174410
  epoch 20 loss 0.4615 best 0.4615 n_mask 174150
  epoch 25 loss 0.4559 best 0.4541 n_mask 175285
  epoch 30 loss 0.4505 best 0.4497 n_mask 174722
  epoch 35 loss 0.4441 best 0.4441 n_mask 175399
  epoch 40 loss 0.4455 best 0.4434 n_mask 174846
  epoch 45 loss 0.4394 best 0.4394 n_mask 174596
  epoch 50 loss 0.4422 best 0.4390 n_mask 175113
masking done best 0.4390


## 7. Finetune Hybrid on curated 645 (load pretrained GNN)


In [ ]:
def train_hybrid(tr_idx, te_idx, tag, r=0.5, epochs=120):
    print(f"\n=== {tag} r={r} train {len(tr_idx)} test {len(te_idx)} ===")
    # class weights
    y_tr=labs[tr_idx]; n=len(y_tr); n1=(y_tr==1).sum(); n0=(y_tr==0).sum()
    w=torch.tensor([n/(2*max(1,n0)), n/(2*max(1,n1))], dtype=torch.float, device=device)
    print(f"weights inactive {w[0]:.2f} active {w[1]:.2f}")
    model=HybridModel(r=r).to(device)
    # load pretrained GNN encoder if exists
    pretrain_path="Part_17/models/gnn_masked_pretrain.pth"
    if os.path.exists(pretrain_path):
        try:
            sd=torch.load(pretrain_path, map_location=device)
            # Hybrid's gnn_enc is GCNEncoder - load compatible keys
            model.gnn_enc.load_state_dict(sd, strict=False)
            print(f"loaded {pretrain_path}")
        except Exception as e: print(f"load failed {e}")
    crit=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
    best_auc=0; best_state=None
    for epoch in range(1, epochs+1):
        model.train(); perm=np.random.permutation(tr_idx)
        total=0
        for s in range(0,len(perm),64):
            bidx=perm[s:s+64]; bgs=[graphs_cur[i] for i in bidx]; bfps=torch.tensor(fps_cur[bidx], dtype=torch.float, device=device)
            loader=DataLoader(bgs, batch_size=len(bgs), shuffle=False)
            batch=next(iter(loader)).to(device)
            opt.zero_grad(); out=model(batch.x, batch.edge_index, batch.batch, bfps); loss=crit(out, batch.y); loss.backward(); opt.step()
            total+=loss.item()*len(bidx)
        if epoch%10==0 or epoch==1:
            model.eval()
            with torch.no_grad():
                sc=[]; tr=[]
                for s in range(0,len(te_idx),128):
                    bidx=te_idx[s:s+128]; bgs=[graphs_cur[i] for i in bidx]; bfps=torch.tensor(fps_cur[bidx], dtype=torch.float, device=device)
                    loader=DataLoader(bgs, batch_size=len(bgs), shuffle=False)
                    batch=next(iter(loader)).to(device); out=model(batch.x,batch.edge_index,batch.batch,bfps); sc.extend(F.softmax(out,dim=1)[:,1].cpu().numpy().tolist()); tr.extend(labs[bidx].tolist())
                roc=roc_auc_score(tr, sc) if len(set(tr))>1 else 0.5
                sched.step(1-roc)
                if roc>best_auc: best_auc=roc; best_state={k:v.cpu() for k,v in model.state_dict().items()}
                print(f"  ep{epoch:3d} loss {total/len(tr_idx):.4f} roc {roc:.4f} best {best_auc:.4f}")
    if best_state: model.load_state_dict({k:v.to(device) for k,v in best_state.items()})
    torch.save(model.state_dict(), f"Part_17/models/hybrid_{tag}_r{r}.pth")
    # final metrics
    model.eval()
    with torch.no_grad():
        sc=[]; tr=[]
        for s in range(0,len(te_idx),128):
            bidx=te_idx[s:s+128]; bgs=[graphs_cur[i] for i in bidx]; bfps=torch.tensor(fps_cur[bidx], dtype=torch.float, device=device)
            loader=DataLoader(bgs, batch_size=len(bgs), shuffle=False)
            batch=next(iter(loader)).to(device); out=model(batch.x,batch.edge_index,batch.batch,bfps); sc.extend(F.softmax(out,dim=1)[:,1].cpu().numpy().tolist()); tr.extend(labs[bidx].tolist())
    sc=np.array(sc); tr=np.array(tr)
    from sklearn.metrics import precision_recall_curve, auc
    def ef(y_true,y_score, frac=0.01):
        n=len(y_true); k=max(1,int(n*frac)); order=np.argsort(-y_score); hits=(y_true[order[:k]]==1).sum(); base=(y_true==1).mean(); return (hits/k)/base if base>0 else 0
    pr, rec,_=precision_recall_curve(tr, sc) if len(set(tr))>1 else ([0],[0],None)
    metrics={"roc_auc":float(roc_auc_score(tr,sc) if len(set(tr))>1 else 0.5), "pr_auc":float(auc(rec,pr) if len(pr)>1 else 0), "f1":float(f1_score(tr,(sc>=0.5).astype(int))), "acc":float(accuracy_score(tr,(sc>=0.5).astype(int))), "mcc":float(matthews_corrcoef(tr,(sc>=0.5).astype(int)) if len(set(tr))>1 else 0), "ef1":float(ef(tr,sc,0.01))}
    print(f"{tag} final", metrics)
    return metrics, sc, tr



In [ ]:
# scratch baseline (no pretrain) for comparison - same function but without load
def train_scratch(tr_idx, te_idx, tag, r=0.5, epochs=120):
    # temporarily hide pretrain file
    import shutil, os
    p="Part_17/models/gnn_masked_pretrain.pth"
    bak=p+".bak" if os.path.exists(p) else None
    if bak: shutil.move(p,bak)
    m,sc,tr=train_hybrid(tr_idx, te_idx, tag+"_scratch", r, epochs)
    # rename file back
    if bak:
        os.rename(f"Part_17/models/hybrid_{tag}_scratch_r{r}.pth", f"Part_17/models/hybrid_{tag}_scratch_temp.pth")
        shutil.move(bak,p)
        os.rename(f"Part_17/models/hybrid_{tag}_scratch_temp.pth", f"Part_17/models/hybrid_{tag}_scratch_r{r}.pth")
    return m,sc,tr



## 8. Run finetune - scaffold + random, with and without pretrain


In [ ]:
metrics={}
m_scaf, sc_scaf, tr_scaf_eval = train_hybrid(tr_scaf, te_scaf, "scaffold", r=0.5, epochs=120)
metrics['scaffold_pretrain']=m_scaf
m_scaf_sc, _, _ = train_scratch(tr_scaf, te_scaf, "scaffold", r=0.5, epochs=120)
metrics['scaffold_scratch']=m_scaf_sc
# random
m_rand, _, _ = train_hybrid(tr_rand, te_rand, "random", r=0.5, epochs=120)
metrics['random_pretrain']=m_rand
m_rand_sc, _, _ = train_scratch(tr_rand, te_rand, "random", r=0.5, epochs=120)
metrics['random_scratch']=m_rand_sc
# RF baseline on scaffold
rf=RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight='balanced')
rf.fit(fps_cur[tr_scaf], labs[tr_scaf]); prob=rf.predict_proba(fps_cur[te_scaf])[:,1]
metrics['scaffold_rf']={"roc_auc":float(roc_auc_score(labs[te_scaf], prob)), "pr_auc":0, "f1":float(f1_score(labs[te_scaf], (prob>=0.5).astype(int))), "acc":float(accuracy_score(labs[te_scaf], (prob>=0.5).astype(int)))}
import pickle
with open("Part_17/models/rf_scaffold.pkl","wb") as f: pickle.dump(rf,f)
print(metrics)
pd.DataFrame(metrics).T.to_csv("Part_17/results/metrics.csv")
print("saved metrics.csv")
pd.DataFrame(metrics).T




=== scaffold r=0.5 train 511 test 134 ===
weights inactive 1.12 active 0.90
loaded Part_17/models/gnn_masked_pretrain.pth
  ep  1 loss 0.5965 roc 1.0000 best 1.0000
  ep 10 loss 0.1030 roc 0.9860 best 1.0000
  ep 20 loss 0.0941 roc 1.0000 best 1.0000
  ep 30 loss 0.0662 roc 1.0000 best 1.0000
  ep 40 loss 0.0479 roc 1.0000 best 1.0000
  ep 50 loss 0.0461 roc 0.9953 best 1.0000
  ep 60 loss 0.0441 roc 1.0000 best 1.0000
  ep 70 loss 0.0542 roc 0.9784 best 1.0000
  ep 80 loss 0.0452 roc 1.0000 best 1.0000
  ep 90 loss 0.0551 roc 1.0000 best 1.0000
  ep100 loss 0.0540 roc 0.9950 best 1.0000
  ep110 loss 0.0431 roc 0.9896 best 1.0000
  ep120 loss 0.0377 roc 0.9856 best 1.0000
scaffold final {'roc_auc': 1.0, 'pr_auc': 1.0, 'f1': 0.6629834254143646, 'acc': 0.5447761194029851, 'mcc': 0.2951472603527322, 'ef1': 2.2333333333333334}

=== scaffold_scratch r=0.5 train 511 test 134 ===
weights inactive 1.12 active 0.90
  ep  1 loss 0.6521 roc 1.0000 best 1.0000
  ep 10 loss 0.0930 roc 0.9890 best 

,roc_auc,pr_auc,f1,acc,mcc,ef1
scaffold_pretrain,1.000000,1.000000,0.662983,0.544776,0.295147,2.233333
scaffold_scratch,1.000000,1.000000,0.779221,0.746269,0.587389,2.233333
random_pretrain,0.972343,0.969751,0.949640,0.945736,0.890932,1.869565
random_scratch,0.973309,0.971662,0.927536,0.922481,0.844203,1.869565
scaffold_rf,1.000000,0.000000,0.991597,0.992537,NaN,NaN


## 9. COCONUT screening - in-domain 38k


In [ ]:
# reuse domain from cell 4 - if no COCONUT, skip screening
if len(coconut)==0 or 'in_domain' not in coconut.columns or coconut['in_domain'].sum()==0:
    print("no in-domain COCONUT -> skipping featurize/screen. Metrics from finetune already saved in Part_17/results/metrics.csv")
    valid_idx=[]
    cocos_fp=np.zeros((0,FP_DIM), dtype=np.float32)
    cocos_graphs=[]
    probs_pre=np.array([])
    probs_scr=np.array([])
    probs_rf=np.array([])
else:
    coconut_with_domain=coconut.copy()
    in_idx=coconut_with_domain[coconut_with_domain['in_domain']].index.tolist()
    print(f"in-domain {len(in_idx)}")
    cocos_fp=[]; cocos_graphs=[]; valid_idx=[]
    for idx in tqdm(in_idx, desc='featurize'):
        smi=coconut_with_domain.at[idx, smiles_col]
        fp=combined_fp(smi); g=mol_to_graph(smi)
        if fp is not None and g is not None:
            cocos_fp.append(fp); cocos_graphs.append(g); valid_idx.append(idx)
    if len(cocos_fp)>0:
        cocos_fp=np.stack(cocos_fp)
        print(f"valid {len(valid_idx)} fp {cocos_fp.shape}")
    else:
        cocos_fp=np.zeros((0,FP_DIM), dtype=np.float32)
        print("no valid featurized")
    # load best models
    def screen(model_path):
        model=HybridModel(r=0.5).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval(); probs=[]
        with torch.no_grad():
            for s in range(0,len(valid_idx),256):
                bfp=torch.tensor(cocos_fp[s:s+256], dtype=torch.float, device=device)
                bgs=cocos_graphs[s:s+256]
                loader=DataLoader(bgs, batch_size=len(bgs), shuffle=False)
                batch=next(iter(loader)).to(device)
                out=model(batch.x,batch.edge_index,batch.batch,bfp)
                probs.extend(F.softmax(out,dim=1)[:,1].cpu().numpy().tolist())
        return np.array(probs)
    if len(valid_idx)>0 and os.path.exists("Part_17/models/hybrid_scaffold_r0.5.pth"):
        probs_pre=screen("Part_17/models/hybrid_scaffold_r0.5.pth")
        probs_scr=screen("Part_17/models/hybrid_scaffold_scratch_r0.5.pth") if os.path.exists("Part_17/models/hybrid_scaffold_scratch_r0.5.pth") else probs_pre
        import pickle
        rf=pickle.load(open("Part_17/models/rf_scaffold.pkl","rb")) if os.path.exists("Part_17/models/rf_scaffold.pkl") else None
        probs_rf=rf.predict_proba(cocos_fp)[:,1] if rf is not None else np.array([])
        print(f"pretrain mean {probs_pre.mean():.4f} max {probs_pre.max():.4f} >0.5 {(probs_pre>0.5).sum()} >0.85 {(probs_pre>0.85).sum()}")
        print(f"scratch  mean {probs_scr.mean():.4f} max {probs_scr.max():.4f} >0.5 {(probs_scr>0.5).sum()} >0.85 {(probs_scr>0.85).sum()}")
        if len(probs_rf)>0:
            print(f"rf       mean {probs_rf.mean():.4f} max {probs_rf.max():.4f} >0.5 {(probs_rf>0.5).sum()}")
        res=coconut_with_domain.loc[valid_idx, ['mw','logp','n_hba','n_hbd']].copy()
        res['canonical_smiles']=coconut_with_domain.loc[valid_idx, smiles_col].values
        res['identifier']=coconut_with_domain.loc[valid_idx, 'identifier'].values if 'identifier' in coconut_with_domain.columns else valid_idx
        res['hybrid_pretrain_prob']=probs_pre; res['hybrid_scratch_prob']=probs_scr; 
        if len(probs_rf)>0: res['rf_prob']=probs_rf
        res.to_csv("Part_17/results/coconut_hybrid_screening.csv", index=False)
        print("saved screening csv")
        for t in [0.5,0.7,0.85,0.9]: print(f"t {t} pre {(probs_pre>t).sum()} scr {(probs_scr>t).sum()} rf {(probs_rf>t).sum() if len(probs_rf)>0 else 0}")
        if len(res)>0:
            print(res[res['hybrid_pretrain_prob']>0.85].sort_values('hybrid_pretrain_prob', ascending=False).head(5).to_string())
    else:
        print("no valid_idx or models not yet trained -> screening deferred")
        probs_pre=np.array([]); probs_scr=np.array([]); probs_rf=np.array([])



in-domain 38265


featurize:   0%|          | 0/38265 [00:00<?, ?it/s][09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:08:08] DEPRECATION WARNING: please use MorganGenerator
[09:

valid 38265 fp (38265, 2215)
pretrain mean 0.5371 max 0.7162 >0.5 27142 >0.85 0
scratch  mean 0.5095 max 0.5341 >0.5 36588 >0.85 0
rf       mean 0.3152 max 0.7233 >0.5 1315
saved screening csv
t 0.5 pre 27142 scr 36588 rf 1315
t 0.7 pre 8 scr 0 rf 1
t 0.85 pre 0 scr 0 rf 0
t 0.9 pre 0 scr 0 rf 0
Empty DataFrame
Columns: [mw, logp, n_hba, n_hbd, canonical_smiles, identifier, hybrid_pretrain_prob, hybrid_scratch_prob, rf_prob]
Index: []


## 10. Save top-k and hits + plots


In [ ]:
for k in [100,500,1000]:
    res.sort_values('hybrid_pretrain_prob', ascending=False).head(k).to_csv(f"Part_17/results/coconut_top{k}_pretrain.csv", index=False)
    res.sort_values('hybrid_scratch_prob', ascending=False).head(k).to_csv(f"Part_17/results/coconut_top{k}_scratch.csv", index=False)
res[res['hybrid_pretrain_prob']>0.85].to_csv("Part_17/results/coconut_hits_pretrain_0.85.csv", index=False)
res[res['hybrid_pretrain_prob']>0.9].to_csv("Part_17/results/coconut_hits_pretrain_0.9.csv", index=False)
print("hits saved")
fig, axes=plt.subplots(2,2, figsize=(12,8))
axes[0,0].hist(res['hybrid_pretrain_prob'], bins=50, alpha=0.6, label='pretrain', color='steelblue')
axes[0,0].hist(res['hybrid_scratch_prob'], bins=50, alpha=0.5, label='scratch', color='coral')
axes[0,0].axvline(0.85,color='red',ls='--'); axes[0,0].set_title('pretrain vs scratch'); axes[0,0].legend()
axes[0,1].scatter(res['rf_prob'], res['hybrid_pretrain_prob'], s=2, alpha=0.2)
axes[0,1].set_xlabel('RF'); axes[0,1].set_ylabel('pretrain'); axes[0,1].set_title(f"corr {res['hybrid_pretrain_prob'].corr(res['rf_prob']):.2f}")
sc=axes[1,0].scatter(res['mw'], res['logp'], c=res['hybrid_pretrain_prob'], cmap='RdYlGn', s=2, alpha=0.5)
axes[1,0].set_xlabel('MW'); axes[1,0].set_ylabel('LogP'); plt.colorbar(sc, ax=axes[1,0])
ths=np.linspace(0,1,21)
axes[1,1].plot(ths, [(res['hybrid_pretrain_prob']>t).sum() for t in ths], label='pretrain', color='steelblue')
axes[1,1].plot(ths, [(res['hybrid_scratch_prob']>t).sum() for t in ths], label='scratch', color='coral')
axes[1,1].plot(ths, [(res['rf_prob']>t).sum() for t in ths], label='RF', color='green')
axes[1,1].set_yscale('log'); axes[1,1].set_xlabel('threshold'); axes[1,1].set_ylabel('hits'); axes[1,1].legend()
plt.tight_layout(); plt.savefig("Part_17/results/screening_analysis.png", dpi=200); plt.savefig("Part_17/results/ablation.png", dpi=200) # reuse
print("saved pngs")



hits saved
saved pngs


## 11. Summary


In [ ]:
print("Part 17 GNN's Morgan complete")
print(f"pool {len(pool_graphs)} pretrain masking best loaded, finetune scaffold done")
print(f"in-domain {len(valid_idx)} screening done - see metrics.csv and screening_analysis.png")



Part 17 GNN's Morgan complete
pool 38776 pretrain masking best loaded, finetune scaffold done
in-domain 38265 screening done - see metrics.csv and screening_analysis.png
